# Qwen3.8-Flash-Next NVFP4 + DFlash drafter on 2x DGX Spark

| Metric | Value | Source |
|---|---|---|
| Decode, c=1, DFlash best block (5) | **52.32** tok/s | definitive sweep, 2026-09-09, 2 boots |
| Decode, c=1, native MTP k=4 (the tuned baseline that matters) | **50.37** tok/s | definitive sweep, 2 boots |
| Decode, c=1, native MTP k=3 (the baseline we used to quote) | **50.07** tok/s | definitive sweep, 2 boots |
| Decode, c=1, AR (spec off) | **24.85** tok/s | definitive sweep, 2 boots |
| Aggregate uplift over the target's own tuned MTP head | **+3.87 %** [+2.10 %, +5.77 %] | paired stratified bootstrap over prompts, method below |
| Same uplift against the softer k=3 baseline | **+4.50 %** [+2.35 %, +6.80 %] | same analysis, k=3 as baseline |

This is an **experimental research artifact with a partly negative result**,
published in the shape it was measured. It is a drafter for *math*; code is a
wash and it regressed *chat* at **every** block length, including the short
blocks that were untested when this notebook was first drafted. The intervals
above carry **no boot-to-boot variance component**, and boot-to-boot movement
is the same size as the effect. Read section 1.2 before you read any number.

```bash
huggingface-cli download nvidia/Qwen3.8-Flash-Next-NVFP4 --revision fc694b54fb0174e0913e6adf86691ef85a4ead47
huggingface-cli download PixelML/Qwen3.8-Flash-Next-NVFP4-DFlash --revision 96758da7
```

Tracking: a private PixelML ticket · Upstream: [sgl-project/sglang#38589](https://github.com/sgl-project/sglang/issues/38589) · Trainer: [deepseek-ai/DeepSpec](https://github.com/deepseek-ai/DeepSpec) (MIT) · Receipts: [results/2026-09-09-.../](../results/2026-09-09-qwen3.8-flash-next-dflash-drafter-2node-tp2-vllm/)


## Qwen3.8-Flash-Next NVFP4 + DFlash drafter — 2x DGX Spark, vLLM TP=2 + EP

Executed notebook. Every cell output below is committed; see
`notebooks/README.md` for the section order and the `LIVE` flag convention.

**What this notebook is.** We trained a DFlash speculative-decoding drafter
for `nvidia/Qwen3.8-Flash-Next-NVFP4` from scratch, because no public drafter
for this target existed, and served it on the Spark pair against the target's
own built-in MTP head. The drafter is not the interesting part. Two other
things are: **neither vLLM nor SGLang can serve this class of drafter on this
target without engine surgery**, and the result is **workload-split** — a
large math gain, a smaller code gain, and a real chat regression.

**What this notebook is not.** It is not a release announcement, and it does
not certify losslessness. Section 1.2 lists, in one place, every claim we are
*not* making and why.


In [1]:
# --- Status cell -------------------------------------------------------
# LIVE = False replays the committed receipts under results/<experiment>/.
# LIVE = True runs the published harness against a running OpenAI-compatible
# endpoint whose base URL comes from the SPARK_ENDPOINT environment variable.
# Never hardcode an endpoint address in this notebook. LIVE mode also needs a
# patched engine, the six-file adapter overlay, and the drafter itself --
# see section 3. This notebook's committed outputs were produced with
# LIVE = False.
import os

EXPERIMENT = "2026-09-09-qwen3.8-flash-next-dflash-drafter-2node-tp2-vllm"
RESULTS_DIR = os.path.join("..", "results", EXPERIMENT)
LIVE = False
SPARK_ENDPOINT = os.environ.get("SPARK_ENDPOINT", "").rstrip("/")

print("LIVE =", LIVE)
print("results dir:", RESULTS_DIR)
if LIVE:
    assert SPARK_ENDPOINT, "Set SPARK_ENDPOINT to your own endpoint base URL before running with LIVE=True."
    print("endpoint: <set via SPARK_ENDPOINT env var, not printed>")


LIVE = False
results dir: ../results/2026-09-09-qwen3.8-flash-next-dflash-drafter-2node-tp2-vllm


In [2]:
# --- Helpers -------------------------------------------------------------
import json
from IPython.display import display, Markdown


def load_receipt(name):
    with open(os.path.join(RESULTS_DIR, name)) as f:
        return json.load(f)


def render_table(headers, rows):
    lines = ["| " + " | ".join(headers) + " |",
             "|" + "|".join(["---"] * len(headers)) + "|"]
    for row in rows:
        lines.append("| " + " | ".join("" if c is None else str(c) for c in row) + " |")
    display(Markdown("\n".join(lines)))


def pending(value, unit=""):
    """Render a result field that the re-benchmark has not filled in yet."""
    if value is None:
        return "**pending**"
    return f"{value}{unit}"


ARMS = load_receipt("arms.json")
PINS = load_receipt("config_pins.json")
FINDINGS = load_receipt("engine_findings.json")
TRAIN = load_receipt("training_and_fidelity.json")
DSPARK = load_receipt("dspark_comparison.json")
print("benchmark status:", ARMS["status"])


benchmark status: final


## 1. TL;DR

**Verdict: experimental. A math drafter that washes on code and regresses chat
at every servable block, on a patched engine, measured in eager mode only.**

- Configuration: 2x DGX Spark (GB10), vLLM, **TP2 + expert parallel**,
  **eager mode**, c=1, greedy (temperature 0, seed 42), non-thinking,
  fixed 256-token outputs, 8k context, `gpu_memory_utilization` 0.75 (never
  exceeded on any arm).
- One frozen 100-prompt fixture (**33 code / 33 math / 34 chat**) used for
  every arm. Arms are separate boots: vLLM resolves the speculative config
  once at boot, so they cannot be interleaved. **Two boots per arm, two runs
  of 100 prompts per boot.**
- The drafter is 5 draft layers, trained block 7, `fc` `[2560, 12800]`,
  498.1 M parameters, trained on 98,470 on-policy target-regenerated samples
  (120.8 M tokens) for 7 epochs.
- **Headline: DFlash block 5 is +3.87 % [+2.10 %, +5.77 %] in aggregate
  output throughput over the tuned native MTP baseline (k=4).** Block 4 is
  +3.26 % [+1.84 %, +4.79 %]. Both intervals exclude zero, so the gain is
  real. It is also small, and smaller than the +4.59 % this notebook carried
  as a placeholder-superseding preliminary.

**Why the baseline moved.** The earlier set compared against native MTP k=3
because that is the setting the model card suggests, and never swept k. This
sweep ran k=1 / 3 / 4 / 6. k=4 is the fastest native arm at 50.371 tok/s;
k=3 is 0.60 % behind it with a **CI spanning zero**, so k=3 and k=4 are
indistinguishable and the honest baseline is "about 50.2". Everything here is
quoted against k=4 because it is the harder number to beat, with the k=3
comparison alongside so the change is traceable.

**What did not move, and what to distrust.** Two boots per arm still cannot
estimate boot-to-boot variance, and the observed movement between boots
(AR +3.4 %, MTP k=4 −2.0 %, DFlash block 4 +1.0 %) is the same order as the
+3.87 % being claimed. That is the weakest part of this result and it is
stated next to every number, not in a footnote.

**Protocol deviation, stated.** The measurement brief asked for ≥ 5 runs × 30
prompts per boot. This sweep ran **2 runs × 100 prompts** per boot instead.
The fixture is fixed-order, so `--limit 30` would have measured the *same* 30
prompts every run (11 code / 10 math / 9 chat) and given per-workload
intervals on 9–11 prompts. For a bootstrap that resamples **prompts** and
averages repeats within prompt, 100 prompts × 2 runs strictly dominates: 100
resampling units instead of 30, the full 33/33/34 strata, and 4 repeats per
prompt across the two boots.


In [3]:
# Key metrics, definitive sweep. Baseline is native MTP k=4: the native head
# was swept (k=1/3/4/6) before any comparison was made. "patched" marks the
# two arms that needed our own K allowlist widened; "boots/runs" is printed
# because two arms are thinner than the rest.
render_table(
    ["Arm", "Block / k", "Aggregate tok/s", "vs MTP k=4", "95% CI", "Boots / runs", "Accepted length", "ms / engine pass"],
    [[a["arm"] + (" *(patched)*" if a["patched"] else "") + ("  **(baseline)**" if a["is_baseline"] else ""),
      a["block"] if a["block"] is not None else (f"k={a['k']}" if a.get("k") else "-"),
      a["aggregate_tok_s"],
      "-" if a["vs_baseline_pct"] is None else f"{a['vs_baseline_pct']:+.2f}%",
      "-" if a["ci95"] is None else f"[{a['ci95'][0]:+.2f}, {a['ci95'][1]:+.2f}]",
      f"{a['boots']} / {a['runs']}",
      a["accepted_length"] if a["accepted_length"] is not None else "-",
      a["ms_per_engine_pass"] if a["ms_per_engine_pass"] is not None else "-"]
     for a in ARMS["arms"]],
)
print("status:", ARMS["status"], "|", ARMS["measured"])
print()
print("BASELINE:", ARMS["baseline"])
print()
print("HEADLINE:", ARMS["headline"]["claim"])
print("          ", ARMS["headline"]["second_best"])
print("          vs k=3:", ARMS["headline"]["against_the_softer_k3_baseline"])
print("          shape:", ARMS["headline"]["workload_shape"])
print()
bv = ARMS["boot_variance_disclosure"]
print("BOOT VARIANCE IS NOT IN THESE INTERVALS:")
print(" ", bv["statement"])
for k, v in bv["observed_movement"].items():
    print(f"   {k}: {v}")
print("  headline effect for comparison:", bv["headline_effect_for_comparison"])
print()
print("THIN AND PATCHED ARMS:")
for k, v in ARMS["thin_and_patched_arms"].items():
    print(f" - {k}: {v}")
print()
print("per-boot aggregate tok/s:")
for a in ARMS["arms"]:
    nm = a["arm"] + (f" block {a['block']}" if a["block"] else (f" k={a['k']}" if a.get("k") else ""))
    print(f"   {nm}: {a['per_boot_aggregate_tok_s']}")


| Arm | Block / k | Aggregate tok/s | vs MTP k=4 | 95% CI | Boots / runs | Accepted length | ms / engine pass |
|---|---|---|---|---|---|---|---|
| AR (spec off) | - | 24.853 | -50.66% | [-51.72, -49.58] | 2 / 3 | - | - |
| native MTP | k=1 | 41.321 | -17.97% | [-19.32, -16.59] | 1 / 2 | 1.8436 | 44.721 |
| native MTP | k=3 | 50.071 | -0.60% | [-1.47, +0.25] | 2 / 4 | 2.9965 | 59.81 |
| native MTP  **(baseline)** | k=4 | 50.371 | - | - | 2 / 4 | 3.3669 | 66.731 |
| native MTP | k=6 | 46.987 | -6.72% | [-7.59, -5.80] | 2 / 4 | 3.8252 | 80.971 |
| native MTP, graph mode | k=4 | 48.37 | -3.97% | [-4.68, -3.26] | 2 / 4 | 3.3537 | 69.189 |
| DFlash *(patched)* | 2 | 46.115 | -8.45% | [-9.47, -7.46] | 2 / 4 | 2.2224 | 48.247 |
| DFlash *(patched)* | 3 | 49.959 | -0.82% | [-1.92, +0.25] | 2 / 4 | 2.5531 | 51.098 |
| DFlash | 4 | 52.015 | +3.26% | [+1.84, +4.79] | 2 / 3 | 2.7562 | 52.931 |
| DFlash | 5 | 52.322 | +3.87% | [+2.10, +5.77] | 2 / 4 | 2.8826 | 55.0 |
| DFlash | 7 | 50.536 | +0.33% | [-1.80, +2.58] | 2 / 4 | 2.9837 | 58.882 |

status: final | 2026-09-09, definitive sweep on the 2x DGX Spark pair

BASELINE: native MTP k=4. The baseline was swept (k=1/3/4/6) before the comparison was made. k=4 is the best native arm; k=3 is 0.60% behind it with a CI spanning zero, so the honest baseline is "about 50.2 at k=3 or k=4" and every figure here is quoted against k=4 because it is the harder number.

HEADLINE: DFlash block 5 beats the tuned native MTP baseline (k=4) by +3.87% in aggregate output throughput, 95% CI [+2.10%, +5.77%].
           DFlash block 4, +3.26% [+1.84%, +4.79%].
          vs k=3: block 5 +4.50% [+2.35%, +6.80%]; the 2026-09-08 single-boot set read +4.59% [+2.14%, +7.06%] against the same baseline.
          shape: a math win (+28.0% at block 5), a code wash (+0.37%, CI spans zero), and a chat regression at every servable block.

BOOT VARIANCE IS NOT IN THESE INTERVALS:
  The intervals below are paired over prompts and contain NO boot-to-boot variance component. Two boots per arm cannot estimate on

In [4]:
p = PINS["pins"]
render_table(["Pin", "Value"], list(p.items()))
print()
print("drafter architecture:")
for k, v in PINS["drafter_architecture"].items():
    print(f"  {k}: {v}")
print()
print("training budget:")
for k, v in PINS["training"].items():
    print(f"  {k}: {v}")


| Pin | Value |
|---|---|
| target_model | nvidia/Qwen3.8-Flash-Next-NVFP4 |
| target_revision | fc694b54fb0174e0913e6adf86691ef85a4ead47 |
| drafter_model | PixelML/Qwen3.8-Flash-Next-NVFP4-DFlash |
| drafter_revision | 96758da7c04e40f843eda749b5a7b343d0eddf96 |
| drafter_visibility | public |
| drafter_safetensors_sha256 | 35a23c17c248ff2e3296e6b78882b6d955af3092498fb7b6c48be1af4bfa971a |
| engine | vLLM source e962733e08d10f7ca65dac4df99e116460b8b174 (arm64 image vllm/vllm-openai@sha256:89dd8f44...6ee39), plus the patches and overlay in section 3 |
| runner | V2 model runner (VLLM_USE_V2_MODEL_RUNNER default); V1 cannot boot this target at all |
| topology | 2 nodes, TP=2 + expert parallel, direct RoCE link, GB10 (sm_121a, arm64), unified memory |
| quantization | NVFP4 routed experts, BF16 attention/shared (target); BF16 drafter |
| kv_cache_dtype | not overridden (engine default) |
| gpu_memory_utilization | 0.75 (policy ceiling for this hardware, never exceeded) |
| compilation | --enforce-eager (graph capture never demonstrated to fit under the 0.75 policy) |
| max_model_len | 8192 |
| speculative_method | dflash, num_speculative_tokens swept over the served blocks in section 2 |
| sampling | temperature 0 (greedy), seed 42, thinking off, max_tokens 256, ignore_eos |
| trainer | deepseek-ai/DeepSpec @ 005e03b81cec38b7da6399833d609ee89a2587f2 (MIT), fork with a qwen4_exp port |


drafter architecture:
  draft_layers: 5
  trained_block_size: 7
  hidden_size: 2560
  intermediate_size: 7680
  q_heads: 24
  kv_heads: 2
  head_dim: 256
  fc_shape: [2560, 12800]
  target_layer_ids: [3, 15, 23, 35, 43]
  mask_token_id: 248077
  parameters: 498106880
  tensors: 58
  dtype: bf16
  stripped_tensors: ['embed_tokens.weight', 'lm_head.weight']
  note: fc consumes five contracted 2560-wide taps (5 x 2560 = 12800). The final hidden state is the distillation target, not a sixth conditioning tap. embed_tokens and lm_head are bound from the target at load.

training budget:
  corpus_rows_regenerated: 99957
  cached_samples: 98470
  cached_tokens: 120776745
  cache_bytes_per_token: 30720
  cache_size_tb: 3.711
  epochs_run: 7
  epochs_planned: 10
  best_checkpoint: step_1344 (epoch 7)
  on_policy: True
  teacher_for_regeneration: nvidia/Qwen3.8-Flash-Next-NVFP4 @ fc694b54, non-thinking, temp 1.0 / top-p 0.95 / top-k 20, max 4096 new tokens


### 1.1 What the artifact is

`PixelML/Qwen3.8-Flash-Next-NVFP4-DFlash` — **public**, with the full model
card carrying the same caveats as this notebook.**

A DFlash block drafter that **replaces** the target's native MTP head (same
verify slot; drafters do not stack with MTP). It ships as **draft layers plus
the `fc` fusion only**: `embed_tokens.weight` and `lm_head.weight` are
stripped and must be **bound from the target at load**. Those bindings were
proved byte-identical to the serving target's tensors, so train-time and
serve-time embeddings are the same bytes (**measured**).

The `fc` projection consumes **five** contracted 2560-wide taps
(5 x 2560 = 12,800) from full-attention/QSA layers `[3, 15, 23, 35, 43]`. The
final hidden state is the **distillation target, not a sixth conditioning
tap** — an earlier description of this checkpoint said "5 taps + last hidden"
and was wrong.


### 1.2 What we are NOT claiming

This section is the point of the notebook. Nothing below is a footnote. Two
items that stood here in the first draft have since been **settled by
measurement** and are restated as results rather than gaps; the rest still
stand, and one of them got worse.

**1. Boot-to-boot variance is not in any interval here, and it moves as much
as the effect.** Two boots per arm cannot estimate a boot variance component.
Observed movement between the two boots of an arm: AR +3.4 %, native MTP k=4
−2.0 %, DFlash block 4 +1.0 %, against a headline effect of +3.87 %. Per-boot
aggregates are printed with the arm table so this can be read directly. This
is the **weakest part of the result**.

**2. Two arms are thin, and two are patched.** Native MTP k=1 is **one boot**
(2 runs) — it is 18 % off the pace, far outside plausible boot variance, so
one boot settles that it is not the optimum, but it is not a two-boot arm.
DFlash block 4 is **3 runs, not 4**. DFlash blocks 2 and 3 are **patched
arms**: they needed a one-line widening of our own K allowlist in
`serving/plugin/dflash_epoch7.py:68` from `(4,5,7)` to `(2,3,4,5,7)`. That
guard is a Pixel ML scope marker, not a vLLM limit and not architectural, but
the label travels with the arms.

**3. Every τ figure is a PROXY, not a measured accepted length.** The offline
evaluator multiplies *marginal* per-position agreement rates against corpus
tokens. That product is **not** the joint probability of an accepted prefix.
The training curve below (2.159 → 3.079) is labelled proxy throughout, and its
numerical closeness to a served accepted length does not validate it. The
number that *was* measured, from the engine's own acceptance counters, is
served accepted length: block 5 **2.883**, block 7 **2.984**, against native
MTP k=4's **3.367** and k=3's **2.997**. Independently cross-checked for
block 5 against the recovered streaming block structure (25,600 tokens in
8,889 engine steps = mean block 2.880). Two differently-trained proxy points
also cannot establish a data-scaling law; an earlier "more data is exhausted"
claim from this lane has been **withdrawn as unsupported**.

**4. Release gates were still not run.** No c=8 or c=16, no soak, no
thinking-mode and no sampling measurements, no long context. Fixed 256-token
outputs with `ignore_eos` establish neither natural-response latency nor
useful-token throughput. Users who want a math accelerator mostly run thinking
mode with sampling, and **the math gain is unmeasured in the mode they would
use**.

**5. The graph-mode comparison is one-sided, and the blocker is ours.** Graph
memory now demonstrably fits (section 2.5), and graphs make the *baseline*
3.97 % slower. But the DFlash half of the A/B cannot be run: our own serving
adapter raises `ValueError("HC attachment requires enforce_eager")` at
`serving/plugin/dflash_epoch7.py:80`, during the draft model's `__init__`.
That is a correctness guard we wrote for the HC tap. It is **not** a vLLM
limitation and **not** a memory failure. Graph-mode throughput for the drafter
is unmeasured, and fixing the guard is our work.

**6. Losslessness is certified — for the accept path, at block 5, greedy,
against a measured floor.** This item used to read "NOT certified". It has
been redone with the controls that were missing (rendered immediately below): common-prefix
replay, block boundaries recovered from streaming chunks, and the check
restricted to accepted-draft positions. Violation rate **0.784 %** against a
same-prefix nondeterminism floor of **0.844 %** over 16,711 positions, with no
concentration at any within-block offset. It is **not** an absolute claim: it
says nothing about sampling, other block sizes, or long context.

**7. The engine is not run-to-run reproducible at c=1, temperature 0**
(section 2.6). Per-step argmax flip hazard is **0.97 %–2.61 %** depending on
arm, which means **almost every 256-token greedy generation differs somewhere
between two runs of the same server**: eight of the eleven arms diverge on
98 % or more of requests. Any gate written on free-running output identity —
including this harness's own `token_agreement ≥ 0.99` — cannot pass and never
could. The ~2.7 % figure this notebook carried earlier was a *teacher-forced*
per-position rate, not this one.

**8. "Fully reproducible" is not claimed.** The drafter serves only on a
patched engine plus a six-file adapter overlay (section 3), and the corpus is
private because its responses are Qwen outputs. Section 5 states exactly which
steps a reader cannot reproduce and why.


In [5]:
# Losslessness, certified. Method first, then the two numbers that carry it.
L = TRAIN["losslessness"]
print("VERDICT:", L["verdict"])
print()
print("METHOD:", L["method"])
print()
render_table(
    ["quantity", "all emitted", "accepted-draft positions only"],
    [["positions checked", L["all_emitted"]["positions_checked"], f"**{L['accepted_draft_positions_only']['positions_checked']}**"],
     ["violation rate", f"{L['all_emitted']['violation_rate']:.3%}", f"**{L['accepted_draft_positions_only']['violation_rate']:.3%}**"],
     ["nondeterminism floor, identical prefix", f"{L['all_emitted']['nondeterminism_floor_same_prefix']:.3%}",
      f"**{L['accepted_draft_positions_only']['nondeterminism_floor_same_prefix']:.3%}**"]],
)
print(L["accepted_draft_positions_only"]["reading"])
print()
print("MISMATCH BY POSITION WITHIN BLOCK (a real bug concentrates at one offset; noise does not):")
bp = L["by_position_within_block"]
render_table(["offset in block"] + list(bp.keys()),
             [["checked"] + [bp[o]["checked"] for o in bp],
              ["violation"] + [f"{bp[o]['violation_rate']:.2%}" for o in bp],
              ["floor"] + [f"{bp[o]['floor_rate']:.2%}" for o in bp]])
print(L["by_position_reading"])
print()
print("THE >12 LOG UNIT MISMATCHES:", L["large_margin_mismatches_explained"])
print()
print("SCOPE:", L["scope"])
print("SUPERSEDES:", L["supersedes"])


VERDICT: CERTIFIED for the accept path at block 5, greedy, against a measured nondeterminism floor.

METHOD: One continuation captured from the DFlash server at block 5 WITH engine-step block boundaries recovered from the streaming chunk boundaries, so every emitted position is labelled accepted-draft or block-final bonus/correction. That SAME continuation was then scored under teacher forcing on the DFlash server and TWICE on a spec-off (AR) server. Every arm therefore sees an IDENTICAL prefix, and AR-replay-1 vs AR-replay-2 gives the pure nondeterminism floor at exactly those positions.



| quantity | all emitted | accepted-draft positions only |
|---|---|---|
| positions checked | 25600 | **16711** |
| violation rate | 2.316% | **0.784%** |
| nondeterminism floor, identical prefix | 2.215% | **0.844%** |

On the positions the accept path is actually responsible for, the drafter disagrees with the spec-off target LESS often than the target disagrees with itself: 0.784% vs a 0.844% floor. SE about 0.07 pp, so this check resolves to roughly 0.2 pp; the 2026-09-08 version was blind below about 2.5%.

MISMATCH BY POSITION WITHIN BLOCK (a real bug concentrates at one offset; noise does not):


| offset in block | 0 | 1 | 2 | 3 | 4 | 5 |
|---|---|---|---|---|---|---|
| checked | 8889 | 6252 | 4199 | 2805 | 1991 | 1464 |
| violation | 2.70% | 2.51% | 2.17% | 1.93% | 1.71% | 1.16% |
| floor | 2.80% | 2.42% | 1.79% | 1.85% | 1.51% | 0.68% |

A real accept-path bug concentrates at one offset; noise does not. There is no concentration anywhere: violation tracks the floor at every offset and declines monotonically.

THE >12 LOG UNIT MISMATCHES: 49 violations exceed 4 log units, maximum 12.0, and ALL 49 sit at the block-final position, the bonus and correction slot, with NONE at an accepted-draft position. The same tail appears when the DFlash server scores its own capture (53 cases, maximum 12.9), so it is a property of teacher-forced replay at the correction slot, not of the accept path. The earlier claim that "all mismatches are near-ties" was false; the correct statement is that the large-margin mismatches live at the slot the accept path does not own.

SCOPE: Certified at block 5 only, greedy, under teacher forcing, against a floor measured on this stack. NOT an absolute losslessness claim, and it says nothing about sampling, other block sizes, or long context.
SUPERSEDES: The 2026-09-08 teacher-forced check, which compar

## 2. Visible results

### 2.1 Serving arms

Definitive sweep, 2026-09-09. Baseline is **native MTP k=4**, chosen by
sweeping the native head rather than inherited from the model card. Every
interval is a paired stratified bootstrap over prompts (8,000 resamples,
repeats averaged within prompt, resampled within workload stratum) on
aggregate output throughput. Per-boot aggregates are printed underneath
because the intervals do **not** contain boot-to-boot variance.


In [6]:
# Per-workload results. On this artifact the workload split IS the result:
# the blended number hides a large math gain, a code wash and a chat
# regression at every block that can be served.
def pct(w):
    if w["vs_baseline_pct"] is None:
        return "-"
    return f"{w['aggregate_tok_s']} ({w['vs_baseline_pct']:+.2f}% [{w['ci95'][0]:+.2f}, {w['ci95'][1]:+.2f}])"


render_table(
    ["Arm", "code (tok/s, vs k=4)", "math (tok/s, vs k=4)", "chat (tok/s, vs k=4)"],
    [[a["arm"] + (f" block {a['block']}" if a["block"] else (f" k={a['k']}" if a.get("k") else ""))
      + (" *(patched)*" if a["patched"] else ""),
      pct(a["per_workload"]["code"]), pct(a["per_workload"]["math"]), pct(a["per_workload"]["chat"])]
     for a in ARMS["arms"]],
)
cr = ARMS["chat_regression_by_block_vs_mtp_k4"]
print("CHAT REGRESSES AT EVERY SERVABLE BLOCK:")
print(" ", cr["note"])
render_table(["Block", "chat vs MTP k=4", "95% CI"],
             [[b, f"{v['chat_pct']:+.2f}%", f"[{v['ci95'][0]:+.2f}, {v['ci95'][1]:+.2f}]"]
              for b, v in cr["blocks"].items()])
print(cr["best_native_chat_arm"])
print()
print("AGAINST THE SOFTER k=3 BASELINE:")
for nm, v in ARMS["vs_native_mtp_k3"].items():
    print(f"  {nm}: aggregate {v['aggregate_pct']:+.2f}% [{v['ci95'][0]:+.2f}, {v['ci95'][1]:+.2f}]"
          f" | math {v['math_pct']:+.2f}% | code {v['code_pct']:+.2f}%"
          f" | chat {v['chat_pct']:+.2f}% [{v['chat_ci95'][0]:+.2f}, {v['chat_ci95'][1]:+.2f}]")
print()
print("fixture:", ARMS["protocol"]["fixture"])
print()
for k, v in ARMS["protocol"]["metric_definitions"].items():
    print(f"{k}: {v}")
print()
print("analysis:", ARMS["protocol"]["analysis"])
print()
print("deviation from brief:", ARMS["protocol"]["deviation_from_brief"])


| Arm | code (tok/s, vs k=4) | math (tok/s, vs k=4) | chat (tok/s, vs k=4) |
|---|---|---|---|
| AR (spec off) | 24.886 (-53.82% [-55.99, -51.53]) | 24.828 (-57.83% [-58.93, -56.68]) | 24.847 (-40.63% [-42.62, -38.63]) |
| native MTP k=1 | 42.116 (-21.85% [-24.69, -18.96]) | 43.135 (-26.74% [-28.25, -25.19]) | 39.013 (-6.78% [-9.20, -4.39]) |
| native MTP k=3 | 52.351 (-2.85% [-4.58, -1.19]) | 56.774 (-3.57% [-4.87, -2.26]) | 43.281 (+3.42% [+2.00, +4.82]) |
| native MTP k=4 | - | - | - |
| native MTP k=6 | 52.169 (-3.19% [-5.05, -1.17]) | 58.525 (-0.60% [-2.14, +1.02]) | 36.488 (-12.81% [-14.00, -11.51]) |
| native MTP, graph mode k=4 | 51.741 (-3.98% [-5.02, -3.00]) | 56.595 (-3.88% [-5.11, -2.64]) | 40.165 (-4.03% [-5.37, -2.69]) |
| DFlash block 2 *(patched)* | 47.243 (-12.33% [-13.73, -10.94]) | 54.592 (-7.28% [-8.71, -5.94]) | 39.284 (-6.13% [-8.10, -4.13]) |
| DFlash block 3 *(patched)* | 51.564 (-4.31% [-5.79, -2.81]) | 63.881 (+8.50% [+6.22, +10.69]) | 40.233 (-3.87% [-5.74, -2.04]) |
| DFlash block 4 | 54.44 (+1.02% [-1.63, +3.95]) | 70.07 (+19.01% [+15.49, +22.62]) | 40.218 (-3.90% [-5.85, -1.93]) |
| DFlash block 5 | 54.085 (+0.36% [-3.17, +4.21]) | 75.385 (+28.04% [+22.80, +33.37]) | 39.382 (-5.90% [-7.96, -3.89]) |
| DFlash block 7 | 52.525 (-2.53% [-6.53, +2.25]) | 75.961 (+29.02% [+20.95, +37.71]) | 37.114 (-11.32% [-13.17, -9.41]) |

CHAT REGRESSES AT EVERY SERVABLE BLOCK:
  Chat regresses at EVERY block that can be served and every interval excludes zero. It does not improve monotonically as the block shrinks: it bottoms out at blocks 3 and 4 near -3.9% and gets worse again at block 2, which also gives up most of the math gain (-7.28%). The "untested where it could pass" possibility is closed: the narrow math-and-code-only release fails on measurement, not on assumption.


| Block | chat vs MTP k=4 | 95% CI |
|---|---|---|
| 2 | -6.13% | [-8.10, -4.13] |
| 3 | -3.87% | [-5.74, -2.04] |
| 4 | -3.90% | [-5.85, -1.93] |
| 5 | -5.90% | [-7.96, -3.89] |
| 7 | -11.32% | [-13.17, -9.41] |

native MTP k=3 at 43.281 tok/s on chat; the drafter tops out near 40.2 and never approaches it.

AGAINST THE SOFTER k=3 BASELINE:
  DFlash block 3: aggregate -0.22% [-1.44, +1.03] | math +12.52% | code -1.50% | chat -7.04% [-8.68, -5.45]
  DFlash block 4: aggregate +3.88% [+2.19, +5.77] | math +23.42% | code +3.99% | chat -7.08% [-9.01, -5.01]
  DFlash block 5: aggregate +4.50% [+2.35, +6.80] | math +32.78% | code +3.31% | chat -9.01% [-11.12, -6.77]

fixture: eval100 — 100 held-out prompts, 33 code / 33 math / 34 chat, stratified from the training corpus held-out split, frozen and identical for every arm

aggregate_tok_s: sum of completion tokens over all requests / sum of wall time over all requests, counted from the final usage object
median_tok_s: median of per-request tok/s across the 100 prompts
accepted_length: engine acceptance counters: accepted draft tokens per engine pass, including the bonus token
ms_per_engine_pass: group wall time / spec draft passes; includes sampling, s

### 2.2 Per-workload results

Rendered above with the arm table, one row per arm, because on this artifact
the workload split *is* the result and a blended number would hide it.

The shape: **a math win, a code wash, a chat regression.** At block 5 against
the tuned baseline, math is **+28.04 %** [+22.80, +33.37], code is **+0.37 %**
with a CI spanning zero, and chat is **−5.90 %** [−7.96, −3.89].

**The narrow release is now closed on measurement, not assumption.** The hope
was that a shorter block would keep the math gain and stop hurting chat. Both
short blocks have now been run. Chat regresses at **every** servable block and
every interval excludes zero; chat does not improve monotonically as the block
shrinks (block 2 is *worse* than blocks 3 and 4 and also gives up most of the
math gain); and chat throughput never approaches native MTP k=3's 43.28 tok/s.

**The drafter only out-accepts the native head on math.** Served accepted
length at block 5 against MTP k=4: code 2.99 vs 3.62, math 4.24 vs 3.98, chat
2.14 vs 2.76. On chat it accepts less while paying a cheaper pass (54.3 ms vs
66.0 ms) and loses anyway.


### 2.3 The 2026-09-08 single-boot set, preserved and superseded

Kept because this repository preserves negative and superseded results rather
than deleting them, and because a reader who saw the earlier version deserves
the correction in the same place as the result. These are **one boot per
arm**, eager, 2 runs x 100 prompts, measured 2026-09-08, against an **untuned**
baseline (native MTP k=3 only, never swept). **They are not the result of this
notebook.**

What moved: the headline went from **+4.59 % [+2.14 %, +7.06 %] against native
MTP k=3** to **+3.87 % [+2.10 %, +5.77 %] against the tuned k=4 baseline**
(and +4.50 % [+2.35 %, +6.80 %] against k=3 on the new data). Per-workload
reporting moved from medians to aggregate throughput with paired CIs. Blocks 2
and 3 went from untested to tested, and still regress chat. The direction of
the correction is down, and both boots are now published per arm.


In [7]:
prelim = ARMS["preliminary_2026_09_08"]
print(prelim["label"].upper())
print()
print("SUPERSEDED BY:", prelim["superseded_by"])
print()
print("WHAT MOVED:", prelim["what_moved"])
print()
render_table(
    ["Arm", "Block", "Aggregate tok/s", "Median tok/s (95% CI)", "Accepted length", "ms / engine pass", "Boots"],
    [[a["arm"], a.get("block") if a.get("block") is not None else "-",
      a["aggregate_tok_s"], f"{a['median_tok_s']} {a['ci95']}",
      a["accepted_length"] if a["accepted_length"] is not None else "-",
      a["ms_per_engine_pass"] if a["ms_per_engine_pass"] is not None else "-",
      a["boots"]] for a in prelim["arms"]],
)
print()
print("per-workload median tok/s (superseded, single boot):")
render_table(["Arm", "code", "math", "chat"],
             [[k, v["code"], v["math"], v["chat"]] for k, v in prelim["per_workload_median_tok_s"].items()])
print()
print("paired stratified bootstrap on the superseded receipts:")
for k, v in prelim["paired_bootstrap"].items():
    print(f"  {k}: {v['delta_pct']:+.2f}%  95% CI [{v['ci95_pct'][0]:+.2f}%, {v['ci95_pct'][1]:+.2f}%]")
print()
print("Do not quote any number in this block. It is kept for the audit trail only.")


SUPERSEDED 2026-09-09 BY THE DEFINITIVE SWEEP ABOVE. PRELIMINARY: SINGLE BOOT PER ARM, 2 RUNS X 100 PROMPTS, EAGER ONLY, UNTUNED BASELINE (NATIVE MTP K=3 ONLY, NEVER SWEPT), MEASURED 2026-09-08. PRESERVED BECAUSE THIS REPOSITORY KEEPS SUPERSEDED RESULTS RATHER THAN DELETING THEM. NOT THE RESULT OF THIS NOTEBOOK.

SUPERSEDED BY: the 2026-09-09 definitive sweep in "arms" above

WHAT MOVED: headline +4.59% [+2.14, +7.06] vs native MTP k=3 -> +3.87% [+2.10, +5.77] vs the tuned k=4 baseline (and +4.50% [+2.35, +6.80] vs k=3 on the new data); one boot per arm -> two; per-workload medians -> per-workload aggregate throughput with paired CIs; blocks 2 and 3 untested -> tested and still regressing chat.



| Arm | Block | Aggregate tok/s | Median tok/s (95% CI) | Accepted length | ms / engine pass | Boots |
|---|---|---|---|---|---|---|
| AR (spec off) | - | 24.302 | 24.295 [24.267, 24.341] | - | - | 2/2 clean |
| native MTP k=3 | - | 49.947 | 51.368 [48.231, 54.32] | 2.993 | 59.87 | 1/1 clean |
| DFlash | 4 | 51.664 | 54.004 [45.885, 60.208] | 2.739 | 52.96 | 1/1 clean |
| DFlash | 5 | 52.239 | 55.475 [47.865, 59.97] | 2.879 | 55.02 | 1/1 clean |
| DFlash | 7 | 50.657 | 52.356 [45.717, 59.643] | 2.988 | 58.77 | 1/1 clean |


per-workload median tok/s (superseded, single boot):


| Arm | code | math | chat |
|---|---|---|---|
| native MTP k=3 | 52.499 | 57.575 | 43.73 |
| DFlash block 4 | 56.76 | 73.48 | 40.86 |
| DFlash block 5 | 58.79 | 78.36 | 37.83 |
| DFlash block 7 | 59.18 | 81.84 | 36.42 |


paired stratified bootstrap on the superseded receipts:
  DFlash block 5 vs native MTP k=3, aggregate: +4.59%  95% CI [+2.14%, +7.06%]
  DFlash block 4 chat, aggregate: -7.47%  95% CI [-9.78%, -5.21%]

Do not quote any number in this block. It is kept for the audit trail only.


### 2.4 Training curve — **proxy** metric, read the label

Held-out set: 496 rows never trained on. **τ here is a proxy** (a product of
marginal per-position agreement rates), not a measured accepted length —
see 1.2 item 2. Position agreement rates are measured; the τ column is what
the evaluator computes from them and needs fixing before it is quoted again.


In [8]:
tc = TRAIN["training_curve"]
print("held-out:", tc["held_out"])
print("definition:", tc["definition"])
render_table(["Epoch", "Step", "pos-1", "pos-2", "pos-4", "pos-7", "tau (PROXY)"],
             [[r["epoch"], r["step"], r["pos1"], r["pos2"], r["pos4"], r["pos7"], r["tau_proxy"]]
              for r in tc["rows"]])
print("label:", tc["label"])
print()
print("PROXY WARNING:", TRAIN["tau_proxy_warning"])
print()
sal = TRAIN["served_accepted_length"]
print("MEASURED served accepted length (this is the number to cite, not tau):")
render_table(["Arm", "served accepted length"], [[k, v] for k, v in sal["by_arm"].items()])
print("block 5 vs the tuned baseline, per workload (DFlash vs native MTP k=4):")
for w, v in sal["per_workload_block5_vs_k4"].items():
    print(f"   {w}: {v}")
print(sal["reading"])
print()
print(sal["note"])
print("label:", sal["label"])


held-out: 496 rows (corpus rows 99,457-99,956), cached identically, never trained on
definition: tau_proxy = 1 + sum_k prod_{j<=k} p_j, where p_j are MARGINAL per-position agreement rates


| Epoch | Step | pos-1 | pos-2 | pos-4 | pos-7 | tau (PROXY) |
|---|---|---|---|---|---|---|
| 1 | 192 | 0.6462 | 0.5075 | 0.3134 | 0.1684 | 2.1585 |
| 2 | 384 | 0.7522 | 0.6226 | 0.4376 | 0.264 | 2.6254 |
| 3 | 576 | 0.7848 | 0.6629 | 0.4893 | 0.3121 | 2.836 |
| 4 | 768 | 0.7989 | 0.6798 | 0.5132 | 0.3405 | 2.9383 |
| 5 | 960 | 0.8074 | 0.6898 | 0.5271 | 0.3572 | 3.0039 |
| 6 | 1152 | 0.8131 | 0.6983 | 0.537 | 0.3686 | 3.0531 |
| 7 | 1344 | 0.8163 | 0.702 | 0.5428 | 0.3752 | 3.0795 |

label: measured (the agreement numbers) / proxy (what tau means)

PROXY WARNING: Every tau figure in this file is a PROXY, not a measured accepted length. The offline evaluator multiplies MARGINAL per-position agreement rates against corpus tokens; that product is not the joint probability of an accepted prefix. Its numerical closeness to a served accepted length does not validate it. Fix the evaluator to compute the joint accepted-prefix probability before quoting tau again.

MEASURED served accepted length (this is the number to cite, not tau):


| Arm | served accepted length |
|---|---|
| DFlash block 2 (patched) | 2.2224 |
| DFlash block 3 (patched) | 2.5531 |
| DFlash block 4 | 2.7562 |
| DFlash block 5 | 2.8826 |
| DFlash block 7 | 2.9837 |
| native MTP k=1 | 1.8436 |
| native MTP k=3 | 2.9965 |
| native MTP k=4 (baseline) | 3.3669 |
| native MTP k=6 | 3.8252 |
| native MTP k=4 graph | 3.3537 |

block 5 vs the tuned baseline, per workload (DFlash vs native MTP k=4):
   code: 2.99 vs 3.62
   math: 4.24 vs 3.98
   chat: 2.14 vs 2.76
The drafter only out-accepts the native head on math. On chat it accepts 2.14 against the head 2.76 while paying a cheaper pass (54.3 ms vs 66.0 ms) and loses anyway.

Measured accepted length from the engine own acceptance counters (1 + accepted/drafts). This is the number to cite rather than the proxy. Independently cross-checked for block 5 against the streaming block structure: 25,600 tokens in 8,889 engine steps = mean block 2.880, against the counter-derived 2.883.
label: measured (2026-09-09, two boots per arm)


### 2.5 Graph mode — memory fits, graphs hurt the baseline, and our own adapter blocks the drafter

Two things this notebook previously listed as unknown are now measured, and
they point the opposite way to what was assumed. Graph memory **does** fit
under the 0.75 unified-memory policy, and graphs make the **baseline slower**,
so the eager-only protocol was not flattering the drafter. The half of the A/B
that is still missing is blocked by an assertion we wrote ourselves.


In [9]:
G = ARMS["graph_mode"]
print("graph memory fits under the 0.75 policy:", G["memory_fit_under_0.75"])
print(" ", G["evidence"])
print()
print(f"native MTP k=4, graph vs eager: {G['mtp_k4_graph_vs_eager_pct']:+.2f}% "
      f"[{G['mtp_k4_graph_ci95'][0]:+.2f}, {G['mtp_k4_graph_ci95'][1]:+.2f}]")
print(" ", G["reading"])
print()
print("DFlash half of the A/B:", G["dflash_graph"])
print("  error:", G["dflash_graph_error"])
print("  raised at:", G["dflash_graph_code_path"])
print(" ", G["dflash_graph_note"])
print()
print("label:", G["label"])


graph memory fits under the 0.75 policy: True
  native MTP k=4 booted TWICE with {"mode":0,"cudagraph_mode":"FULL_DECODE_ONLY","cudagraph_capture_sizes":[1,2,4,8,16]} at utilization 0.75 and served 4 full runs. This is the FIRST demonstration that graph memory fits under the policy; the earlier attempt died inside profile_cudagraph_memory -> initialize_kv_cache, before capture.

native MTP k=4, graph vs eager: -3.97% [-4.68, -3.26]
  Graphs make the BASELINE slower, consistently across both boots and all three workloads. So the eager-only protocol was NOT flattering the drafter; the earlier speculation that eager inflates per-pass overhead and favours a one-parallel-pass drafter is not supported, and if anything graphs would widen the drafter margin.

DFlash half of the A/B: BLOCKED
  error: ValueError("HC attachment requires enforce_eager")
  raised at: serving/plugin/dflash_epoch7.py:80, raised in the draft model __init__ during load
  This is OUR OWN adapter's assertion, written as 

### 2.6 The engine is not run-to-run reproducible at c=1, temperature 0

This is unflattering to the whole setup and it is the honest resolution floor
for every fidelity check in this notebook, so it is a section rather than a
footnote. Measured same-boot, free-running, on identical prompts, with a
common-prefix estimator: the rate at which two runs first disagree given that
they have agreed so far.


In [10]:
N = TRAIN["engine_nondeterminism"]
print("measurement:", N["measurement"])
print()
hz = N["per_step_argmax_flip_hazard"]
dv = N["sequences_that_diverge_at_all"]
render_table(["Arm", "per-step argmax flip hazard", "sequences that diverge at all"],
             [[k, f"{hz[k]:.2%}", f"{dv[k]:.1%}"] for k in hz])
print(N["reading"])
print()
print("label:", N["label"])


measurement: Same-boot, free-running, identical prompts, with a common-prefix estimator: the rate at which two runs first disagree given they have agreed so far.



| Arm | per-step argmax flip hazard | sequences that diverge at all |
|---|---|---|
| AR (spec off) | 2.61% | 99.0% |
| native MTP k=1 | 1.78% | 98.0% |
| native MTP k=3 | 2.08% | 98.5% |
| native MTP k=4 | 0.97% | 81.0% |
| native MTP k=6 | 1.06% | 80.5% |
| native MTP k=4 graph | 1.15% | 83.0% |
| DFlash block 2 | 2.36% | 98.5% |
| DFlash block 3 | 2.51% | 98.0% |
| DFlash block 4 | 2.29% | 100.0% |
| DFlash block 5 | 2.15% | 98.5% |
| DFlash block 7 | 2.27% | 98.0% |

The engine is NOT run-to-run reproducible at c=1, temperature 0. At roughly 2% per step, almost every 256-token greedy generation differs somewhere between two runs of the same server: eight of the eleven arms diverge on 98% or more of requests, and the three best-behaved still diverge on about 81%. The consequence, which was never stated before: any gate written on free-running output identity cannot pass and never could, including the harness own token_agreement >= 0.99. A ~2.7% figure carried in earlier notes was a TEACHER-FORCED per-position rate, not this one. If you are building acceptance gates for speculative decoding, measure your engine against itself first.

label: measured


## 3. Serving — it will not run without this

**Nothing in this section is optional.** On the pinned engine, a
DeepSpec-trained DFlash drafter for this target does not serve at all. Four
independent code paths block it, and the fix is **three control-flow patches
on top of a six-file adapter overlay**. The patches alone serve nothing: the
overlay is what makes the target emit contracted taps in the first place.

Labels: **measured** = a boot or a run · **source-supported** = read at the
cited file:line.


### 3.1 The four blockers and the three patches


In [11]:
print("label key:", FINDINGS["label_key"])
print()
render_table(
    ["#", "What blocks it", "Error", "Where", "Fix", "Label"],
    [[b["id"], b["what"], f"`{b['error']}`", f"`{b['where']}`", b["fix"], b["label"]]
     for b in FINDINGS["blockers"]],
)
print()
for b in FINDINGS["blockers"]:
    print(f"[{b['id']}] why: {b['why']}")
    print()


label key: measured = a boot or a run; source-supported = read at the cited file:line; inferred; untested



| # | What blocks it | Error | Where | Fix | Label |
|---|---|---|---|---|---|
| 1.1 | V1 runner cannot prepare PLE inputs, so the target does not boot at all — with or without a drafter | `RuntimeError: PLE inputs were not prepared` | `vllm/models/qwen4_exp/nvidia/model.py:300 (raised during profile_run)` | use the V2 runner; there is no V1 path | measured (1 boot) |
| 1.2 | V2 runner dereferences a None MTP hidden-state getter | `TypeError: 'NoneType' object is not subscriptable` | `vllm/v1/worker/gpu/model_runner.py:834 (and the mirrored site at :2016-2018)` | PATCH 1 (control flow only, 2 sites): guard with `if pre_hc_hidden_states is not None`, taking the engine's own absent-attribute fallback | measured (1 boot) |
| 1.3 | the DFlash speculator hard-rejects the anchor layout DeepSpec checkpoints train with | `ValueError: sample_from_anchor=True is not supported for DFlash` | `vllm/v1/worker/gpu/spec_decode/dflash/speculator.py:69` | PATCH 2 (control flow only, :66-75): honour dflash_config.sample_from_anchor and set num_query_per_req = num_speculative_steps when it is set. NOT needed for DSpark, which sets both attributes itself after super().__init__ | measured (1 boot) + source-supported |
| 1.4 | the Qwen4Exp speculative-method allowlist omits our methods | `NotImplementedError, raised before weights load` | `vllm/model_executor/models/config.py, Qwen4ExpForConditionalGenerationConfig.verify_and_update_config` | PATCH 3 (one line each): add "dflash" (and "dspark" if serving that checkpoint) to the allowlist | measured |


[1.1] why: query_start_loc and ngram_context come only from Qwen4ExpModelState.prepare_inputs / prepare_dummy_inputs (qwen4_exp/nvidia/model_state.py:93,112), called only from the V2 runner (v1/worker/gpu/model_runner.py:1772, gpu/cudagraph_utils.py:572)

[1.2] why: the runner overrides the drafter's input hidden states with the target's pre-HC MTP residual whenever the target exposes get_mtp_target_hidden_states(). Qwen4Exp exposes the accessor but returns None when no native MTP head is loaded (method=dflash), and the subscript is unguarded

[1.3] why: vLLM's DFlash path assumes the speculators-format 1+N layout (position 0 is the anchor, its output discarded). DeepSpec trains anchor-as-first-prediction: K query slots, every position predicts. DSparkSpeculator already selects our layout and the shared Triton _prepare_dflash_inputs_kernel already implements it behind a SAMPLE_FROM_ANCHOR constexpr — the raise is the only thing in the way

[1.4] why: upstream the allowlist is {mtp, ng

### 3.2 The six-file adapter overlay the patches sit on


In [12]:
ov = FINDINGS["overlay"]
print(ov["note"])
print("overlay size:", ov["size"])
render_table(["File", "What it adds"], [[f"`{f['file']}`", f["adds"]] for f in ov["files"]])
print()
print("tap mapping:", ov["tap_mapping"])
print("label:", ov["label"])


The three patches above sit ON TOP OF a six-file adapter overlay that makes the target emit contracted taps at all and teaches the proposer the DeepSpec anchor convention. The patches alone serve nothing. Reproducing this needs the whole set.
overlay size: 183 lines


| File | What it adds |
|---|---|
| `vllm/models/qwen4_exp/nvidia/model.py` | captures block_input (the native-width 2560 contracted residual, tuple slot [1]) at boundary layers; exposes set_dflash_aux_hidden_state_layers; declares supports_eagle3 / has_own_embed_tokens=False / has_own_lm_head=False; returns (sample_hidden_states, aux_hidden_states) and raises RuntimeError if fewer than 5 taps are collected |
| `vllm/v1/spec_decode/dflash.py` | query_zero_predicts_next, making num_query_per_req = K (DeepSpec) instead of K + 1 (legacy) |
| `vllm/v1/spec_decode/utils.py` | threads QUERY_ZERO_PREDICTS_NEXT into the Triton kernel: sample_offset = 0 instead of 1, so query 0's output is sampled rather than discarded |
| `vllm/v1/spec_decode/llm_base_proposer.py` | registers Qwen4ExpForConditionalGeneration in the target-sharing allowlist so the draft aliases the target's embedding and untied lm_head |
| `vllm/config/speculative.py` | returns num_draft_tokens - 1 additional scheduler slots when query_zero_predicts_next is set |
| `vllm/model_executor/models/config.py` | the dflash allowlist entry (blocker 1.4) |


tap mapping: The vLLM aux boundary ids are the trained tap ids + 1: trained target_layer_ids [3,15,23,35,43] are captured at [4,16,24,36,44], each at layers[i].attn_hyper_connection.mix/combine_and_mix(...)[1], plus hyper_connection_mixer.combine_and_mix(...)[1] for the final hidden. There is no separate final norm to add, and the learned contraction must never be replaced by a raw HC-stream mean.
label: source-supported (nvidia/model.py:276-331,548-561; nvidia/hyperconnection.py:128-188; DeepSpec modeling/dspark/qwen3/modeling.py:238-246,449-460) + measured (the boots)


### 3.3 Launching an arm

Every arm is a separate boot. vLLM resolves the speculative configuration
once in `VllmConfig` at boot; `SamplingParams` carries no speculative field
and there is no per-request or per-endpoint override. That is also why
per-workload enablement (speculate on math and code, fall back to native MTP
on chat) is a **deployment note** — two server processes with a router in
front — and not a product feature.

Flags that matter, reconstructed from the lane's receipts (the two-node
launcher wrapper, node addressing and mounts are not published, per
`AGENTS.md`):

```bash
# AR baseline: no --speculative-config at all.
vllm serve <local-path-to>/Qwen3.8-Flash-Next-NVFP4 \
  --served-model-name qwen3.8-flash-next \
  --tensor-parallel-size 2 --enable-expert-parallel \
  --max-model-len 8192 --gpu-memory-utilization 0.75 \
  --enforce-eager

# native MTP baseline, k = 3 (and k = 4 for the second baseline arm)
  --speculative-config '{"method": "mtp", "num_speculative_tokens": 3}'

# DFlash drafter, served block 5 (sweep 2/3/4/5/7 as separate boots)
  --speculative-config '{"method": "dflash",
                         "model": "<local-path-to>/Qwen3.8-Flash-Next-NVFP4-DFlash",
                         "num_speculative_tokens": 5,
                         "draft_tensor_parallel_size": 2}'
```

Two things about the drafter side:

- the drafter's `config.json` must carry `sample_from_anchor: true` in its
  `dflash_config`, which is exactly what patch 2 stops the engine from
  rejecting;
- **served block K is a configuration, never an extrapolation.** A fully
  accepted iteration emits K + 1 tokens (K drafts plus the bonus), and the
  drafter was *trained* at block 7 — serving it at 4 or 5 is a different
  operating point that has to be measured, not derived. In the preliminary
  set the trained block was **not** the fastest served block.


### 3.4 The benchmark harness

Two published scripts under
`results/2026-09-09-qwen3.8-flash-next-dflash-drafter-2node-tp2-vllm/`:

- **`run_served_eval.py`** — one arm, one boot. Frozen fixture, c=1, greedy,
  `max_tokens=256`, `ignore_eos`, thinking off. Completion tokens come from
  the **final usage object**, never from stream-event count; accepted length
  comes from the engine's **own acceptance counters** via `/metrics`, not
  from a client-side inference. The endpoint comes from `SPARK_ENDPOINT`;
  no address is hardcoded.
- **`analyze_paired_bootstrap.py`** — the difference and the interval.
  Resamples **prompts**, keeps both arms' records for a resampled prompt
  together, averages repeats within a prompt, stratifies by workload so every
  resample keeps the 33/33/34 mix, and differences the **aggregates**. This
  is the correction to the earlier pooled-median reading. The interval it
  produces is **conditional on the boots that were run** and excludes
  boot-to-boot variability.

```bash
SPARK_ENDPOINT=http://<your-host>:8000/v1 \
python run_served_eval.py --fixture eval100.jsonl --arm dflash-b5 \
    --out DFLASH_B5/run1.json --repeats 2

python analyze_paired_bootstrap.py --a MTP_K3/run1.json --b DFLASH_B5/run1.json \
    --workload chat --resamples 4000
```


In [13]:
# The two published scripts, shown from the receipts directory so a reader can
# see exactly what produced the numbers rather than a paraphrase of it.
for name in ("run_served_eval.py", "analyze_paired_bootstrap.py"):
    path = os.path.join(RESULTS_DIR, name)
    with open(path) as f:
        src = f.read()
    print(f"--- {name}: {len(src.splitlines())} lines ---")
    print("\n".join(src.splitlines()[:24]))
    print("...")
    print()


--- run_served_eval.py: 145 lines ---
#!/usr/bin/env python3
"""Serve-side benchmark harness for one speculative-decoding arm.

This is the published, sanitized form of the protocol used for every arm in
this notebook: one frozen 100-prompt fixture, concurrency 1, greedy, fixed
256-token outputs, counted from the engine's final usage object.

It is deliberately engine-agnostic: point it at any OpenAI-compatible
endpoint. The endpoint base URL comes from SPARK_ENDPOINT; never hardcode an
address here.

One arm = one server boot with one speculative configuration. vLLM resolves
the speculative config once in VllmConfig at boot; there is no per-request
override, so arms cannot be interleaved against one server.

Usage:
    SPARK_ENDPOINT=http://<your-host>:8000/v1 \
    python run_served_eval.py --fixture eval100.jsonl --arm dflash-b5 \
        --out DFLASH_B5/run1.json --repeats 2
"""
from __future__ import annotations

import argparse
import json
...

--- analyze_paired_bootstrap.py: 82

## 4. Related findings

Three results that do not depend on the drafter's throughput number, and are
likely to outlive it.

### 4.1 DSpark: this configuration loses — which is not the same as the method losing

We also trained and served a DSpark drafter (same target, same taps, same
cache, same held-out split, same harness and fixture). It is **worse than
DFlash on every workload** and lands **below the target's own MTP head**.

Two scoping statements, both load-bearing:

1. This is **measured for this configuration in vLLM today**. It is **not**
   evidence that the DSpark method loses. The cost mechanism is **unprofiled**.
2. The explanation this lane published earlier — "five sequential full-vocab
   projections, just batch them" — was **wrong**. The pinned implementation
   already batches the main vocabulary projection; the sequential part is the
   rank-256 Markov bias, which depends on previously sampled tokens. The
   correction is recorded here because the wrong version was published first.

### 4.2 Adaptive verification is structurally incompatible with GatedDeltaNet backends

DSpark's confidence head is real, trained, and **already consumed end to end
by vLLM** — nothing needed implementing. It cannot run here anyway:

```
ValueError: Adaptive verification trims verification requests on device, which the
GDNAttentionBackend attention backend does not support.
  vllm/v1/worker/gpu/spec_decode/adaptive_verification.py:463
  <- maybe_create_adaptive_verification_manager
  <- vllm/v1/worker/gpu/model_runner.py:614 initialize_kv_cache
```

`get_query_lens_mismatch_unsupported_backend`
(`v1/worker/gpu/attn_utils.py:190-208`) rejects any backend whose
`supports_device_cpu_query_lens_mismatch()` is False. `GDNAttentionBackend`
is one, and Qwen3.8-Flash-Next has **36 GatedDeltaNet layers**, so the check
can never pass on this target. Adaptive verification needs to shrink
per-request query lengths *on device* after the CPU batch is built, and a
Mamba-class linear-attention backend cannot honour that. **This generalises
to every Mamba-class hybrid, not just this model** — the widest-blast-radius
finding in this notebook. (**measured**, the boot; **source-supported**, the
mechanism.)

Note also there is no threshold to sweep:
`AdaptiveVerificationManager.get_num_tokens:331` picks
`argmax(expected_accepted_tokens / cost)` against profiled cost curves — a
cost-model optimum, not a cutoff.

### 4.3 SGLang cannot serve this class of drafter on this target at all

Desk check, **source-supported, nothing measured**. SGLang `main` (fetched
2026-09-08) and the pinned cookbook image
`d91c3682b0b429e4c70df63cd57f819588ce29b0` agree on every point; line numbers
are `main`, the pinned image is +12 in `models/qwen4_exp.py`.

- The cookbook's `DFLASH` string for this model is a **UI label map, not a
  capability**. The per-model speculative option list has three entries:
  `current`, `off (greedy)`, and `mtp` = `NEXTN`. All ~20 hardware cells,
  **both DGX Spark cells included**, bake `NEXTN` and nothing else. The page
  is not wrong; reading a shared playground helper as a per-model capability
  was.
- Root cause: **`qwen4_exp` cannot emit DFlash aux hidden states**, and setup
  **succeeds silently**. `_init_qwen4_exp_layer_extensions:1222-1228` deletes
  `layer_communicator`; neither decoder layer consumes
  `captured_last_layer_outputs` (the identifier occurs exactly once in the
  file, at the call site `:1658`); and `Qwen4ExpModel.forward:1655-1662`
  returns `(hidden_states, hc_hidden_states)`, leaving the aux return
  reachable only in the idle branch. The DFlash worker therefore receives the
  **HC-flattened 4 x 2560 = 10240-wide** stream instead of concatenated
  per-layer taps.
- The fix has a precedent: `glm5_next.py:953-962` (`hc_contract`, sglang
  #36708) solved exactly this for its own multi-HC model.
- One safety asymmetry worth naming: on the anchor-layout mismatch vLLM
  **raises**; SGLang would **silently** shift every draft position by one and
  merely look like a bad drafter.

Filed read-only upstream as
[sgl-project/sglang#38589](https://github.com/sgl-project/sglang/issues/38589).
No PR, no port started.


In [14]:
print(DSPARK["label"])
print()
print("SCOPE:", DSPARK["scope_warning"])
print()
render_table(["Workload", "native MTP k=3 (tok/s)", "DSpark block 5 (tok/s)", "DSpark vs MTP"],
             [[w,
               DSPARK["per_workload_median_tok_s"]["native MTP k=3"][w],
               DSPARK["per_workload_median_tok_s"]["DSpark block 5"][w],
               f"{DSPARK['per_workload_vs_mtp_pct'][w]:+.1f}%"]
              for w in ["math", "code", "chat"]])
print()
print("aggregate:", DSPARK["aggregate"])
print("accepted length by workload:", DSPARK["accepted_length_by_workload"])
print()
print("held-out proxy comparison:", DSPARK["held_out_proxy"]["note"])
print()
print("confidence head status:", DSPARK["confidence_head"]["status"])


measured, preliminary (single boot, same harness/fixture/protocol as the DFlash arms, 2026-09-08)

SCOPE: These numbers describe THIS DSpark configuration in vLLM today. They are not evidence that the DSpark method loses. The cost mechanism is unprofiled, and an earlier published explanation ('five sequential full-vocab projections, just batch them') was WRONG: the pinned implementation already batches the main vocabulary projection, and the sequential part is the rank-256 Markov bias, which depends on previously sampled tokens.



| Workload | native MTP k=3 (tok/s) | DSpark block 5 (tok/s) | DSpark vs MTP |
|---|---|---|---|
| math | 57.575 | 69.506 | +20.7% |
| code | 52.499 | 48.604 | -7.4% |
| chat | 43.73 | 32.3 | -26.1% |


aggregate: {'DSpark block 5 tok/s': 44.446, 'second run': 44.221, 'median_tok_s': 46.895, 'ci95': [40.525, 50.845], 'vs_native_mtp_k3_pct': -11.0}
accepted length by workload: {'math': 3.94, 'code': 2.653, 'chat': 1.896}

held-out proxy comparison: the reference +16-18% accepted-length gain over DFlash did not reproduce on this target and corpus; proxy tau is a wash. DSpark leads DFlash at every position from pos-3 onward and trails at pos-1 (0.7659 vs 0.8163), consistent with its 10% CE / 90% L1 loss mix.

confidence head status: real, trained, and correctly wired in vLLM end to end — and unreachable on this target, because adaptive verification cannot run on a GatedDeltaNet backend. Never calibrated (no predicted-vs-realised curve, no chat-vs-math distribution).


In [15]:
for r in FINDINGS["related"]:
    print("###", r["id"])
    print(r.get("what", ""))
    if "error" in r:
        print(r["error"])
        print("  at", r["where"])
    if "why" in r:
        print(r["why"])
    if "detail" in r:
        print(r["detail"])
    if "upstream" in r:
        print("upstream:", r["upstream"])
    print("label:", r["label"])
    print()


### adaptive-verification-vs-gdn
vLLM's adaptive verification is structurally incompatible with GatedDeltaNet backends, so DSpark's confidence gate cannot run on this target at all
ValueError: Adaptive verification trims verification requests on device, which the GDNAttentionBackend attention backend does not support.
  at vllm/v1/worker/gpu/spec_decode/adaptive_verification.py:463 <- maybe_create_adaptive_verification_manager <- v1/worker/gpu/model_runner.py:614 initialize_kv_cache
get_query_lens_mismatch_unsupported_backend (v1/worker/gpu/attn_utils.py:190-208) rejects any backend whose supports_device_cpu_query_lens_mismatch() is False. GDNAttentionBackend is one, and Qwen3.8-Flash-Next has 36 GatedDeltaNet layers, so the check can never pass. This generalises to every Mamba-class hybrid target, not just this model.
label: measured (the boot) + source-supported (the mechanism)

### graph-capture-not-demonstrated
graph capture was never demonstrated to fit under the 0.75 unified-memo

## 5. Reproduce

### 5.1 Hardware

2x DGX Spark (GB10, `sm_121a`, arm64), one GPU per node, ~120 GiB unified
memory per node, direct RoCE link. TP=2 with expert parallel across the pair.
`gpu_memory_utilization` 0.75 is a **policy ceiling for this hardware**, not a
tuning choice: it was never exceeded, and a host-memory floor watcher stopped
any boot that approached it.

### 5.2 Pins

Printed in section 1 from `config_pins.json`. The load-bearing ones:

| | |
|---|---|
| target | `nvidia/Qwen3.8-Flash-Next-NVFP4@fc694b54fb0174e0913e6adf86691ef85a4ead47` |
| drafter | `PixelML/Qwen3.8-Flash-Next-NVFP4-DFlash@96758da7c04e40f843eda749b5a7b343d0eddf96` — **public** |
| engine | vLLM source `e962733e08d10f7ca65dac4df99e116460b8b174`, arm64 image `vllm/vllm-openai@sha256:89dd8f44…6ee39`, **plus the three patches and the six-file overlay in section 3** |
| runner | V2 model runner. V1 cannot boot this target at all |
| trainer | `deepseek-ai/DeepSpec@005e03b81cec38b7da6399833d609ee89a2587f2` (MIT), forked for a `qwen4_exp` port |

### 5.3 The fixture

`eval100`: **100 prompts, 33 code / 33 math / 34 chat**, a stratified subset
of the 500-row held-out split of the training corpus (corpus rows
99,457–99,956), frozen and byte-identical across every arm. First user turn
only. 256 tokens per request with `ignore_eos`, so every arm decodes exactly
the same number of tokens.

**This file is not published.** Its prompts descend from eight Apache-2.0 /
MIT `open-perfectblend` sources, but the corpus it was cut from carries Qwen
model outputs as its responses, so the whole corpus is governed by the Qwen
Community License 1.0 and the NVIDIA Open Model License and is held private
pending a licence read. What is published is the recipe: seed 42, DeepSpec's
`download_and_split.py --test-size 0.05 --seed 42`, stratify by `source`,
take the held-out split, keep the 33/33/34 workload mix. Any fixture built
that way is comparable in shape, not in bytes.

### 5.4 The analysis, exactly

Paired stratified bootstrap over prompts, 4,000 resamples, implemented in
`analyze_paired_bootstrap.py`: resample prompts (stratified by workload),
keep both arms' records for a resampled prompt together, average repeats
within a prompt, then difference the **aggregate** throughputs. Report the
percentage difference with a **conditional** 95 % CI — conditional on the
boots that were run, excluding boot-to-boot variability and any selection
over block sizes.

**Do not** compare two independently bootstrapped pooled-median intervals and
read overlap as "no difference". That is not a paired test, and it produced a
wrong conclusion on this very lane.

**Do not** compare across harnesses. Re-measuring the native MTP baseline on
this fixture moved it from 41.9 to 49.947 tok/s; publishing against the old
number would have overstated the drafter by roughly 16 %. Cross-harness
comparison is barred in this lane.

### 5.5 What you cannot reproduce publicly, and why

| step | status |
|---|---|
| the drafter weights | published; the corpus behind them is not |
| the training corpus and the `eval100` fixture | private — responses are Qwen model outputs (Qwen Community License 1.0 + NVIDIA Open Model License); a licence read is required before any public flip |
| the engine patches and the six-file overlay | described here at file:line with the exact control-flow change; the patch files themselves live in the lane receipts, not in this repo |
| the two-node launcher wrapper, node addressing, mounts, raw logs | not published, per `AGENTS.md` |
| everything else — protocol, fixture composition, harness, analysis, pins, blockers | published here and runnable against your own endpoint and your own drafter |

### 5.6 Licences and attribution

- **Trainer / method.** `deepseek-ai/DeepSpec` and the DFlash reference
  implementation are **MIT**; their notices travel with any derivative of the
  training code. Inco's DFlash2 checkpoints (CC-BY-NC-ND) were deliberately
  **not** used — DFlash and DSpark are the releasable paths.
- **The drafter checkpoint.** It was trained on **outputs of a Qwen model**,
  so it is governed by the **Qwen Community License 1.0** and, because the
  serving teacher is NVIDIA's NVFP4 conversion, the **NVIDIA Open Model
  License**. MIT trainer licensing does not resolve artifact-release
  obligations.
- **Attribution.** "Built with Qwen." The target is
  `nvidia/Qwen3.8-Flash-Next-NVFP4`, itself derived from
  `Qwen/Qwen3.8-Flash-Next`.
- **Prompts.** The eight `open-perfectblend` sources are Apache-2.0 / MIT and
  are credited in the corpus manifest.


## 6. Appendix

<details>
<summary>Failure history, corrections, and what each one cost</summary>

### Boots that failed, in order

| attempt | outcome |
|---|---|
| V1 runner + DFlash | `RuntimeError: PLE inputs were not prepared` (`qwen4_exp/nvidia/model.py:300`). The target cannot boot under V1 at all, with or without a drafter — which is why the V1 `spec_decode` overlays passed CPU dry-runs and never ran on GPU |
| V2 unpatched + DFlash | `TypeError: 'NoneType' object is not subscriptable` (`model_runner.py:834`) |
| native MTP k=3 without the upstream MTP overlay | failed on block-FP8 MTP weight names |
| DSpark adaptive, `--no-eager` (inductor) | host MemAvailable 49.4 → 0.7 GiB inside one 10 s watcher sample; the 12 GiB floor watcher stopped the container (exit 137, `OOMKilled=false`) |
| DSpark adaptive, `--no-eager --compilation-config '{"mode":0,...}'` | `ValueError` at `adaptive_verification.py:463` — GDN backend, inside `profile_cudagraph_memory → initialize_kv_cache`, before any capture |
| five-file adapter overlay | rejected at `EngineArgs.create_engine_config` before weights load; a sixth guard (the spec-method allowlist) was required |
| DFlash + graph capture, 2026-09-09 | `ValueError: HC attachment requires enforce_eager` at `serving/plugin/dflash_epoch7.py:80`, in the draft model `__init__`. **Our own guard**, not vLLM and not memory. It is why the graph A/B is one-sided |
| DFlash blocks 2 and 3, first attempt | rejected by our own K allowlist at `serving/plugin/dflash_epoch7.py:68`, which listed `(4,5,7)`. Widened to `(2,3,4,5,7)` for those two arms; they are labelled patched wherever they appear |

### Corrections this lane published and then had to withdraw

Recorded because they were published first, and a reader who saw the earlier
version deserves the retraction in the same place as the result.

| claim | status |
|---|---|
| "lossless under the teacher-forced criterion" | **withdrawn as a certification on 2026-09-08, then re-established properly on 2026-09-09.** The 2026-09-08 check covered all emitted tokens including correction/bonus tokens, replayed different prefixes across arms, and could not explain mismatch margins above 12 log units. The redone check uses common-prefix replay, recovers block boundaries from streaming chunks, and restricts to accepted-draft positions: **0.784 % violation against a 0.844 % same-prefix floor over 16,711 positions.** The 12-log-unit tail is explained — all 49 such cases sit at the block-final correction slot, none at an accepted-draft position, and the same tail appears in the self-control. The accept path is now **certified at block 5, greedy, against a measured floor**, and nothing broader |
| "τ 3.079 / the training curve / the data-scaling slope" | **withdrawn as measurement.** The evaluator multiplies marginal per-position rates, which is not a joint accepted-prefix probability. Everything downstream of that metric is a proxy, including the "+0.48 τ per 10× data" slope and the 1.9 M-sample extrapolation that was used to cancel further data rounds |
| "the gain is indistinguishable from zero" | **wrong.** It came from comparing pooled per-request medians when the headline is aggregate throughput. The paired stratified bootstrap on the same receipts showed a real, small gain |
| "block-4 chat is inside noise, the narrow release is untested where it could pass" | **wrong in the other direction, and now closed on measurement.** The paired bootstrap put block-4 chat at −7.47 % [−9.78 %, −5.21 %] against k=3 on the old data. On the definitive sweep, chat regresses at **every** servable block against the tuned k=4 baseline (2: −6.13 %, 3: −3.87 %, 4: −3.90 %, 5: −5.90 %, 7: −11.32 %), every interval excluding zero, and it does **not** improve monotonically as the block shrinks. The blocks that could have saved the narrow release were tested and did not |
| "a graph-mode A/B is feasible, the `mode:0` boot cleared compile" | **wrong then, and the follow-up correction is ours too.** That boot did fail inside `profile_cudagraph_memory → initialize_kv_cache`, before capture. On 2026-09-09 graph memory **was** demonstrated to fit under the 0.75 policy (native MTP k=4, two boots, four runs), and graphs make that baseline **3.97 % slower**, so the earlier speculation that eager mode might be flattering the drafter is **not supported**. The DFlash half still cannot be run, and the blocker is **our own** adapter assertion at `serving/plugin/dflash_epoch7.py:80`, not vLLM and not memory |
| "DSpark pays five sequential full-vocab projections; just batch them" | **wrong.** The pinned implementation already batches the main vocabulary projection; the sequential part is the rank-256 Markov bias, which depends on previously sampled tokens. The real cost is unprofiled |
| "5 taps + last hidden feed the fusion" | **wrong.** `fc` in_features is 5 x 2560 = 12,800; the last hidden state is the distillation target, not a sixth conditioning tap |
| "≥ 25 % over tuned MTP" as the bar | **dropped.** It was set early and arbitrarily. Changing an acceptance criterion after the fact is stated explicitly rather than quietly: the original criterion was **not met**, and this notebook publishes the artifact as experimental rather than re-labelling a miss as a pass |
| "the MTP baseline is tuned" | **corrected, and it did win.** Only k=3 had been re-measured on this fixture. The definitive sweep ran k=1/3/4/6: k=4 is the fastest native arm at 50.371 tok/s. It moved the headline from +4.59 % to +3.87 %. k=3 and k=4 are statistically a tie (0.60 %, CI spanning zero), so the honest baseline is "about 50.2" |
| "the engine's ~2.7 % argmax flip rate" | **mis-labelled.** That was a *teacher-forced* per-position rate. Measured free-running with a common-prefix estimator, the per-step flip hazard is 0.97 %–2.61 % depending on arm, and the consequence that was never stated is that almost every 256-token greedy generation differs between two runs of the same server. Any gate on free-running output identity above the floor cannot pass, this harness's own `token_agreement ≥ 0.99` included |

### Measurement bugs worth stealing

- **`prompt_logprobs=0` measures the wrong thing.** An exporter fidelity check
  scored 0.8302 with `prompt_logprobs=0`, which returns only the *actual*
  prompt token's logprob — so it was measuring teacher-versus-corpus
  agreement. With `prompt_logprobs=1`, which also returns the engine's argmax,
  the same check scored **0.99695**. The 0.83 figure said nothing about the
  exporter.
- **Shrink the epochs, not the depth.** A cheap "3 draft layers, 1 epoch"
  diagnostic returned position-1 agreement 0.0482 and would have stopped the
  lane. At *equal* training on the same cache, 5 layers gave 0.3010 — a 6x
  gap. A 3-layer draft is below the useful threshold for this target, not a
  scaled-down proxy for the 5-layer one.
- **Forward hooks silently never fire under `@support_torch_compile`.** The
  decorator installs its own `__call__`, so `nn.Module._call_impl` never runs.
  Observed as "20/20 prompts processed at 15,640 tok/s, 0 captures" — a
  failure that produces an empty cache with a green exit code.
- **The HF reference implementation of this architecture is unusable for
  feature extraction:** ~38 tok/s on 4x B200, ~95 % of forward time inside
  the QSA indexer's nested Python loop. Features were exported from vLLM
  instead — faster, and *more correct*, since vLLM's hidden states are the
  ones the drafter sees at serve time (11,899 tok/s on one B200).
- **Never train more than one epoch without intermediate checkpoints.** A
  cancelled run at step 760/770 with only an end-of-training save lost the
  entire run.

### A recorded deviation, not an oversight

The exporter fidelity gate was specified as per-tap cosine ≥ 0.98 against a
BF16 HF reference. The worst tap measured **0.9761** and the lane proceeded
deliberately, on three grounds: the *internal* check (`lm_head(last_hidden)`
argmax reproduces the engine's own top-1 on **0.99695** of 9,194 positions)
cannot be affected by quantization; vLLM and HF reproduce the same inter-tap
geometry to 3–4 decimals; and the gap decays monotonically with depth, the
signature of quantization drift rather than a mis-hooked module. The residual
gap is NVFP4-versus-BF16, and in that comparison the BF16 path is the
approximation — the NVFP4 engine generated the corpus and serves the drafter.
Recorded here in the same words as in the model card.

### Cost and limits

Training the drafter cost on the order of a few hundred dollars of rented
B200 time end to end (regeneration, a 3.7 TB feature cache, and 7 epochs);
the serving measurements ran on owned hardware. The reusable artifact is not
the checkpoint: it is the on-policy corpus and the tap fixture, which are
teacher-agnostic on the prompt side and let a future lane check teacher
fidelity in one boot instead of re-deriving it.

### Review history

This notebook is the output of an adversarial review round, and the reviews
disagreed with each other. One review argued the gain was indistinguishable
from zero and the chat regression might be rescuable at a smaller block; a
second review, run against the receipts with the statistics re-run, showed
the first was wrong on both counts — the gain was real and small, and the
chat regression was real at every block tested. That second review also found
the two errors nobody else had: that τ was never measured, and that the
published DSpark cost explanation was false. Both are corrected above. The
lane's own prediction ledger records that **eight of eleven** recorded
predictions were wrong.

</details>
